# IQL Sweep Results Tables

Collects results from W&B sweeps and formats them into paper-style tables
(rows = datasets, columns = methods).

- **Table 1 — Performance:** for every run we take the **max** of the eval
  `mean_score` over training, then report **mean ± error** of those per-run
  maxima across the sweep (one run per seed).
- **Table 2 — Steps-to-goal:** for every run we find the eval step where
  `mean_score` is maximal and read `avg_steps_to_goal` at that same step, then
  report **mean ± std** across runs.

The sweeps listed below populate the **"IQL with task reward"** column. Add
more sweep IDs to `SWEEPS` as new results come in and re-run the notebook.

> **Scaling note.** `iql.py` logs `mean_score` as the raw evaluation return.
> For antmaze the per-episode reward is sparse (1 on reaching the goal, else 0),
> so `mean_score` is the success rate in `[0, 1]`. We multiply by `SCALE = 100`
> to match the 0–100 scale shown in the paper table. `avg_steps_to_goal` is a
> raw step count and is **not** scaled.

In [ ]:
# !pip install wandb pandas numpy  # uncomment if needed
import time

import numpy as np
import pandas as pd
import wandb

api = wandb.Api()

## Configuration

- `ENTITY`: your W&B entity (team/user). Leave as `""` to use your default
  entity from `wandb login`.
- `PROJECT`: the W&B project the sweeps live in.
- `METRIC`: the performance metric to maximize per run.
- `STEPS_METRIC`: the steps-to-goal metric read at the best-`METRIC` step.
- `SCALE`: multiplier applied to `METRIC` (100 turns an antmaze success rate
  into a 0–100 score). Not applied to `STEPS_METRIC`.
- `ERROR_TYPE`: what the `±` shows — `"std"`, `"var"`, or `"sem"`.
- `DDOF`: delta-degrees-of-freedom for std/var (`1` = sample std, `0` =
  population std).
- `DROP_FAILED`: for Table 2, exclude runs whose best-`METRIC` step had no
  successful episodes (`avg_steps_to_goal == -1`), since a -1 step count is
  not meaningful.

In [ ]:
ENTITY = "champlin-university-of-arizona"  # "" = default entity from wandb login
PROJECT = "IQL-pref"
METRIC = "mean_score"
STEPS_METRIC = "avg_steps_to_goal"
SCALE = 100.0
ERROR_TYPE = "std"  # "std" | "var" | "sem"
DDOF = 1  # 1 = sample std/var (recommended across seeds)
DROP_FAILED = True  # Table 2: drop runs that never solved (steps == -1)

## Sweep registry

`SWEEPS[dataset][method] = sweep_id`. Method keys map to table columns via
`METHOD_COLUMNS` below. Fill in more IDs as you get results — empty entries
render as blank cells.

In [ ]:
SWEEPS = {
    "antmaze-medium-play-v2": {
        "task_reward": "567akajk",
        "MR": "s5ji4y63",
        "PT": "24nt68im",
    },
    "antmaze-medium-diverse-v2": {
        "task_reward": "new66902",
        "MR": "e68vsrhs",
        "PT": "0fy3y45g",
    },
    "antmaze-large-play-v2": {
        "task_reward": "0ua455rr",
        "MR": "inh37i1n",
    },
    "antmaze-large-diverse-v2": {
        "task_reward": "dzzhw2wg",
        "MR": "ccimkc3n",
    },
}

# (method_key, (column group, column subheader)) — controls table layout/order.
METHOD_COLUMNS = [
    ("task_reward", ("IQL with task reward", "")),
    ("MR", ("IQL with preference learning", "MR")),
    ("PT", ("IQL with preference learning", "PT")),
]

## Shared helpers

`run.history(...)` (the **sampled-history** endpoint, one request per run) is
used rather than `run.scan_history(...)`, which paginates through *every*
logged training step and is much slower. `samples=100000` is far above the
number of eval points, so per-run maxima / argmax lookups are exact. Per-run
progress is printed so a slow sweep is visibly progressing, not hung.

In [ ]:
def _sweep_runs(sweep_id):
    path = f"{ENTITY}/{PROJECT}/{sweep_id}" if ENTITY else f"{PROJECT}/{sweep_id}"
    return list(api.sweep(path).runs)


def summarize(values, ddof=DDOF):
    arr = np.asarray(values, dtype=float)
    n = arr.size
    if n == 0:
        return None
    std = float(arr.std(ddof=ddof)) if n > ddof else 0.0
    var = float(arr.var(ddof=ddof)) if n > ddof else 0.0
    return {
        "mean": float(arr.mean()),
        "std": std,
        "var": var,
        "sem": std / np.sqrt(n) if n else 0.0,
        "n": n,
        "values": arr,
    }


def fmt(stats, err=ERROR_TYPE):
    if stats is None or stats["n"] == 0:
        return ""
    return f"{stats['mean']:.2f} ± {stats[err]:.2f}"


def build_table(results):
    columns = pd.MultiIndex.from_tuples([col for _, col in METHOD_COLUMNS])
    rows = list(SWEEPS.keys())
    tbl = pd.DataFrame(index=rows, columns=columns, dtype=object)
    tbl.index.name = "Dataset"
    for dataset in rows:
        for method, col in METHOD_COLUMNS:
            tbl.loc[dataset, col] = fmt(results.get(dataset, {}).get(method))
    tbl.fillna("", inplace=True)
    return tbl


def build_detail(results):
    out = []
    for dataset, methods in SWEEPS.items():
        for method, sweep_id in methods.items():
            stats = results.get(dataset, {}).get(method)
            if not stats:
                continue
            out.append(
                {
                    "dataset": dataset,
                    "method": method,
                    "sweep_id": sweep_id,
                    "n_runs": stats["n"],
                    "mean": round(stats["mean"], 2),
                    "std": round(stats["std"], 2),
                    "var": round(stats["var"], 2),
                    "sem": round(stats["sem"], 2),
                    "values": np.round(np.sort(stats["values"])[::-1], 2).tolist(),
                }
            )
    return pd.DataFrame(out)

# Table 1 — Performance (max `mean_score`)

For each run, the max eval `mean_score` over training (scaled by `SCALE`).

In [ ]:
_cache = {}


def _run_max(run, metric=METRIC):
    """Max of `metric` for one run via the fast sampled-history endpoint."""
    df = run.history(keys=[metric], samples=100000, pandas=True)
    if df is None or metric not in getattr(df, "columns", []):
        return None
    series = df[metric].dropna()
    return float(series.max()) if not series.empty else None


def fetch_run_max_scores(
    sweep_id, metric=METRIC, scale=SCALE, force=False, verbose=True
):
    """Return a list of per-run max(metric) values (scaled) for one sweep."""
    if sweep_id in _cache and not force:
        return _cache[sweep_id]
    runs = _sweep_runs(sweep_id)
    if verbose:
        print(f"  sweep {sweep_id}: {len(runs)} run(s)")
    maxima, t0 = [], time.time()
    for i, run in enumerate(runs, 1):
        m = _run_max(run, metric)
        if m is not None:
            maxima.append(scale * m)
        if verbose:
            shown = f"{scale * m:.2f}" if m is not None else "no data"
            print(f"    [{i}/{len(runs)}] {run.name} ({run.state}): {shown}")
    if verbose:
        print(f"  -> {len(maxima)} run(s) with data in {time.time() - t0:.1f}s")
    _cache[sweep_id] = maxima
    return maxima


results = {}
for dataset, methods in SWEEPS.items():
    results[dataset] = {}
    for method, sweep_id in methods.items():
        if not sweep_id:
            continue
        print(f"{dataset}  [{method}]")
        results[dataset][method] = summarize(fetch_run_max_scores(sweep_id))
        print(f"  => {fmt(results[dataset][method])}\n")

In [ ]:
table = build_table(results)
table

### Per-run detail (performance)

In [ ]:
build_detail(results)

# Table 2 — Average steps-to-goal at best `mean_score`

For each run we locate the eval step where `mean_score` is maximal, then read
`avg_steps_to_goal` at that same step (both metrics are logged together each
eval). We report **mean ± std** across runs.

`avg_steps_to_goal` is logged by `iql.py` only for antmaze, as the mean
steps-to-goal over *successful* episodes that eval, or `-1` when none reached
the goal. A run whose best `mean_score` is 0 (never solved) therefore reads
`-1`; with `DROP_FAILED = True` those runs are excluded here. The number
excluded is printed.

In [ ]:
_steps_cache = {}


def _run_steps_at_best(run, metric=METRIC, steps_metric=STEPS_METRIC):
    """`steps_metric` at the step where `metric` is maximal, for one run."""
    df = run.history(keys=[metric, steps_metric], samples=100000, pandas=True)
    cols = getattr(df, "columns", [])
    if df is None or metric not in cols or steps_metric not in cols:
        return None
    sub = df[[metric, steps_metric]].dropna(subset=[metric])
    if sub.empty:
        return None
    best_step = sub[metric].idxmax()  # first step attaining the max
    val = sub.loc[best_step, steps_metric]
    return None if pd.isna(val) else float(val)


def fetch_run_steps_at_best(sweep_id, force=False, verbose=True):
    """Per-run steps-to-goal at the best-`METRIC` step for one sweep."""
    if sweep_id in _steps_cache and not force:
        return _steps_cache[sweep_id]
    runs = _sweep_runs(sweep_id)
    if verbose:
        print(f"  sweep {sweep_id}: {len(runs)} run(s)")
    vals, dropped, t0 = [], 0, time.time()
    for i, run in enumerate(runs, 1):
        s = _run_steps_at_best(run)
        keep = s is not None and not (DROP_FAILED and s < 0)
        if keep:
            vals.append(s)
        elif s is not None:
            dropped += 1
        if verbose:
            if s is None:
                tag = "no data"
            elif keep:
                tag = f"{s:.2f}"
            else:
                tag = f"{s:.2f} (dropped: never solved)"
            print(f"    [{i}/{len(runs)}] {run.name} ({run.state}): {tag}")
    if verbose:
        print(
            f"  -> {len(vals)} run(s) kept, {dropped} dropped in {time.time() - t0:.1f}s"
        )
    _steps_cache[sweep_id] = vals
    return vals


steps_results = {}
for dataset, methods in SWEEPS.items():
    steps_results[dataset] = {}
    for method, sweep_id in methods.items():
        if not sweep_id:
            continue
        print(f"{dataset}  [{method}]")
        steps_results[dataset][method] = summarize(fetch_run_steps_at_best(sweep_id))
        print(f"  => {fmt(steps_results[dataset][method])}\n")

In [ ]:
steps_table = build_table(steps_results)
steps_table

### Per-run detail (steps-to-goal)

In [ ]:
build_detail(steps_results)

## Export

Copy-pasteable Markdown and LaTeX for both tables.

In [ ]:
for name, tbl in [
    ("Table 1 — performance", table),
    ("Table 2 — steps-to-goal", steps_table),
]:
    print(f"## {name}\n")
    print("### Markdown\n")
    print(tbl.to_markdown())
    print("\n### LaTeX\n")
    print(tbl.to_latex(multicolumn=True, multicolumn_format="c"))
    print()
# table.to_csv("results_table.csv")
# steps_table.to_csv("results_steps_table.csv")